### Step 1 : Import the required libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import hstack

### Step 2 : Load the Dataset

In [ ]:
df = pd.read_csv('/content/Dataset .csv')

In [ ]:
df

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,Botswana Pula(P),Yes,No,No,No,4,4.4,Green,Very Good,270
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,Botswana Pula(P),No,No,No,No,4,4.9,Dark Green,Excellent,365
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.584450,"Japanese, Korean",...,Botswana Pula(P),Yes,No,No,No,4,4.8,Dark Green,Excellent,229
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9546,5915730,Naml۱ Gurme,208,��stanbul,"Kemanke�� Karamustafa Pa��a Mahallesi, R۱ht۱m ...",Karak�_y,"Karak�_y, ��stanbul",28.977392,41.022793,Turkish,...,Turkish Lira(TL),No,No,No,No,3,4.1,Green,Very Good,788
9547,5908749,Ceviz A��ac۱,208,��stanbul,"Ko��uyolu Mahallesi, Muhittin ��st�_nda�� Cadd...",Ko��uyolu,"Ko��uyolu, ��stanbul",29.041297,41.009847,"World Cuisine, Patisserie, Cafe",...,Turkish Lira(TL),No,No,No,No,3,4.2,Green,Very Good,1034
9548,5915807,Huqqa,208,��stanbul,"Kuru�_e��me Mahallesi, Muallim Naci Caddesi, N...",Kuru�_e��me,"Kuru�_e��me, ��stanbul",29.034640,41.055817,"Italian, World Cuisine",...,Turkish Lira(TL),No,No,No,No,4,3.7,Yellow,Good,661
9549,5916112,A���k Kahve,208,��stanbul,"Kuru�_e��me Mahallesi, Muallim Naci Caddesi, N...",Kuru�_e��me,"Kuru�_e��me, ��stanbul",29.036019,41.057979,Restaurant Cafe,...,Turkish Lira(TL),No,No,No,No,4,4.0,Green,Very Good,901


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9551 entries, 0 to 9550
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Restaurant ID         9551 non-null   int64  
 1   Restaurant Name       9551 non-null   object 
 2   Country Code          9551 non-null   int64  
 3   City                  9551 non-null   object 
 4   Address               9551 non-null   object 
 5   Locality              9551 non-null   object 
 6   Locality Verbose      9551 non-null   object 
 7   Longitude             9551 non-null   float64
 8   Latitude              9551 non-null   float64
 9   Cuisines              9542 non-null   object 
 10  Average Cost for two  9551 non-null   int64  
 11  Currency              9551 non-null   object 
 12  Has Table booking     9551 non-null   object 
 13  Has Online delivery   9551 non-null   object 
 14  Is delivering now     9551 non-null   object 
 15  Switch to order menu 

### Step 3 : Data Preprocessing

1. Feature Selection - Select relevant columns / features for Recommending the Restaurant

In [ ]:
df_rec = df[[
    "Restaurant Name",
    "Cuisines",
    "Average Cost for two",
    "Price range",
    "Has Online delivery",
    "Has Table booking",
    "Country Code"
]]

In [ ]:
df_rec.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9551 entries, 0 to 9550
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Restaurant Name       9551 non-null   object
 1   Cuisines              9542 non-null   object
 2   Average Cost for two  9551 non-null   int64 
 3   Price range           9551 non-null   int64 
 4   Has Online delivery   9551 non-null   object
 5   Has Table booking     9551 non-null   object
 6   Country Code          9551 non-null   int64 
dtypes: int64(3), object(4)
memory usage: 522.4+ KB


2. Handling the Missing Values

In [ ]:
df_rec.isnull().sum()

,0
Restaurant Name,0
Cuisines,9
Average Cost for two,0
Price range,0
Has Online delivery,0
Has Table booking,0
Country Code,0


In [ ]:
df_rec["Cuisines"] = df_rec["Cuisines"].fillna("Unknown")

/tmp/ipython-input-2483695547.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_rec["Cuisines"] = df_rec["Cuisines"].fillna("Unknown")


In [ ]:
df_rec.isnull().sum()

,0
Restaurant Name,0
Cuisines,0
Average Cost for two,0
Price range,0
Has Online delivery,0
Has Table booking,0
Country Code,0


### Step 4 : Categorical Value Encoding

In [ ]:
df_rec

,Restaurant Name,Cuisines,Average Cost for two,Price range,Has Online delivery,Has Table booking,Country Code
0,Le Petit Souffle,"French, Japanese, Desserts",1100,3,No,Yes,162
1,Izakaya Kikufuji,Japanese,1200,3,No,Yes,162
2,Heat - Edsa Shangri-La,"Seafood, Asian, Filipino, Indian",4000,4,No,Yes,162
3,Ooma,"Japanese, Sushi",1500,4,No,No,162
4,Sambo Kojin,"Japanese, Korean",1500,4,No,Yes,162
...,...,...,...,...,...,...,...
9546,Naml۱ Gurme,Turkish,80,3,No,No,208
9547,Ceviz A��ac۱,"World Cuisine, Patisserie, Cafe",105,3,No,No,208
9548,Huqqa,"Italian, World Cuisine",170,4,No,No,208
9549,A���k Kahve,Restaurant Cafe,120,4,No,No,208


In [ ]:
df_rec["Has Online delivery"] = df_rec["Has Online delivery"].map({"Yes": 1, "No": 0})

/tmp/ipython-input-3888059766.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_rec["Has Online delivery"] = df_rec["Has Online delivery"].map({"Yes": 1, "No": 0})


In [ ]:
df_rec["Has Table booking"] = df_rec["Has Table booking"].map({"Yes": 1, "No": 0})

/tmp/ipython-input-705522188.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_rec["Has Table booking"] = df_rec["Has Table booking"].map({"Yes": 1, "No": 0})


In [ ]:
df_rec

,Restaurant Name,Cuisines,Average Cost for two,Price range,Has Online delivery,Has Table booking,Country Code
0,Le Petit Souffle,"French, Japanese, Desserts",1100,3,0,1,162
1,Izakaya Kikufuji,Japanese,1200,3,0,1,162
2,Heat - Edsa Shangri-La,"Seafood, Asian, Filipino, Indian",4000,4,0,1,162
3,Ooma,"Japanese, Sushi",1500,4,0,0,162
4,Sambo Kojin,"Japanese, Korean",1500,4,0,1,162
...,...,...,...,...,...,...,...
9546,Naml۱ Gurme,Turkish,80,3,0,0,208
9547,Ceviz A��ac۱,"World Cuisine, Patisserie, Cafe",105,3,0,0,208
9548,Huqqa,"Italian, World Cuisine",170,4,0,0,208
9549,A���k Kahve,Restaurant Cafe,120,4,0,0,208


### Step 5 : Content Based Filtering Approach

1. Vectorize Cuisines

In [ ]:
tfidf = TfidfVectorizer(stop_words="english")
cuisine_matrix = tfidf.fit_transform(df_rec["Cuisines"])

In [ ]:
cuisine_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 27059 stored elements and shape (9551, 149)>

2. Normalize Numric Features

In [ ]:
scaler = MinMaxScaler()

In [ ]:
numeric_features = df_rec[
    ["Average Cost for two", "Price range",
     "Has Online delivery", "Has Table booking"]
]

In [ ]:
numeric_scaled = scaler.fit_transform(numeric_features)

3. Combine Cuisine + Numeric Features

In [ ]:
restaurant_features = hstack([
    cuisine_matrix,
    numeric_scaled
])

In [ ]:
restaurant_features

<COOrdinate sparse matrix of dtype 'float64'
	with 45308 stored elements and shape (9551, 153)>

### Step 6 : Testing the Recommendation System

1. Define Sample User Preferences

In [ ]:
user_preferences = {
    "cuisines": "Chinese, Asian",
    "price_range": 3,
    "average_cost_for_two": 1500,
    "online_delivery": 0,
    "table_booking": 1
}

2. Generate Recommendations

In [ ]:
user_cuisine_vec = tfidf.transform([user_preferences["cuisines"]])

In [ ]:
user_numeric_vec = scaler.transform([[
    user_preferences["average_cost_for_two"],
    user_preferences["price_range"],
    user_preferences["online_delivery"],
    user_preferences["table_booking"]
]])

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [ ]:
user_vector = hstack([user_cuisine_vec, user_numeric_vec])

In [ ]:

similarity_scores = cosine_similarity(
    user_vector, restaurant_features
).flatten()

In [ ]:
df_rec["Similarity Score"] = similarity_scores

/tmp/ipython-input-3383870959.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_rec["Similarity Score"] = similarity_scores


In [ ]:
top_recommendations = df_rec.sort_values(
    by="Similarity Score",
    ascending=False
).head(5)

In [ ]:
top_recommendations[
    ["Restaurant Name", "Cuisines", "Price range", "Average Cost for two", "Similarity Score"]
]

,Restaurant Name,Cuisines,Price range,Average Cost for two,Similarity Score
5017,Diva Spiced,"Asian, Chinese",4,2000,0.984732
4978,Chinois,Asian,3,1500,0.959939
2076,Flying Tuk Tuk,Asian,3,1100,0.959939
5006,Ping's Caf�� Orient,Asian,4,2100,0.948570
7542,Spicy Duck - Taj Palace Hotel,Asian,4,4000,0.948569
